# 03 — Analysis & Visualizations
**Goal:** Answer key questions about Medicaid spending using the exported Parquet files.

**Key Questions:**
1. How has total Medicaid spending trended 2018–2024?
2. Which states spend the most per beneficiary?
3. What are the top 20 costliest HCPCS procedures?
4. Who are the top 50 providers by total paid?
5. Which providers have anomalously high billing rates? (fraud signals)

In [ ]:
import duckdb
import polars as pl
import plotly.express as px
import matplotlib.pyplot as plt
import os, json
from pathlib import Path

con = duckdb.connect()

# ── Load paths from shared config ─────────────────────────────────────────────
_config_path = Path(__file__).parent.parent / ".medicaid_config.json"
_cfg = {}
if _config_path.exists():
    _cfg = json.loads(_config_path.read_text())

exports_dir = _cfg.get("exports_dir", str(Path(__file__).parent.parent / "exports"))
print("Exports dir:", exports_dir)

# ── Load pre-exported parquet files ───────────────────────────────────────────
_missing = [f for f in ["by_state_month.parquet", "by_hcpcs_month.parquet", "top_providers.parquet"]
            if not os.path.exists(os.path.join(exports_dir, f))]
if _missing:
    raise FileNotFoundError(
        f"Missing export files: {_missing}\n"
        "Run notebook 02_cleaning.ipynb first to generate these files."
    )

by_state = con.sql(f"SELECT * FROM read_parquet('{os.path.join(exports_dir, 'by_state_month.parquet')}')").pl()
by_hcpcs = con.sql(f"SELECT * FROM read_parquet('{os.path.join(exports_dir, 'by_hcpcs_month.parquet')}')").pl()
providers = con.sql(f"SELECT * FROM read_parquet('{os.path.join(exports_dir, 'top_providers.parquet')}')").pl()

print(f"by_state : {by_state.shape}")
print(f"by_hcpcs : {by_hcpcs.shape}")
print(f"providers: {providers.shape}")
print("All exports loaded successfully")

## 1. Spending Trend Over Time

In [ ]:
df_time = (
    by_state.group_by("year", "month")
    .agg(pl.col("total_paid").sum())
    .sort(["year", "month"])
    .with_columns((pl.col("year") + "-" + pl.col("month")).alias("period"))
    .to_pandas()
)

fig = px.line(
    df_time,
    x="period",
    y="total_paid",
    title="Total Medicaid Spending Over Time",
    labels={"total_paid": "Total Paid ($)", "period": "Month"},
)
fig.show()

## 2. Spending by State (Total)

In [ ]:
df_state = (
    by_state.group_by("state")
    .agg(pl.col("total_paid").sum(), pl.col("total_beneficiaries").sum())
    .with_columns(
        (pl.col("total_paid") / pl.col("total_beneficiaries")).alias(
            "paid_per_beneficiary"
        )
    )
    .sort("total_paid", descending=True)
    .to_pandas()
)

fig = px.choropleth(
    df_state,
    locations="state",
    locationmode="USA-states",
    color="total_paid",
    scope="usa",
    title="Total Medicaid Spending by State",
    color_continuous_scale="Blues",
)
fig.show()

## 3. Top 20 Costliest HCPCS Procedures

In [ ]:
df_hcpcs = (
    by_hcpcs.group_by("HCPCS_CODE", "hcpcs_description")
    .agg(pl.col("total_paid").sum())
    .sort("total_paid", descending=True)
    .head(20)
    .to_pandas()
)

fig = px.bar(
    df_hcpcs,
    x="total_paid",
    y="HCPCS_CODE",
    orientation="h",
    hover_data=["hcpcs_description"],
    title="Top 20 HCPCS Procedures by Total Spend",
    labels={"total_paid": "Total Paid ($)", "HCPCS_CODE": "Code"},
)
fig.update_layout(yaxis={"categoryorder": "total ascending"})
fig.show()

## 4. Top 50 Providers by Total Paid

In [ ]:
top50 = providers.head(50).to_pandas()

fig = px.scatter(
    top50,
    x="total_claims",
    y="total_paid",
    hover_data=["npi", "last_name", "state"],
    size="total_beneficiaries",
    color="state",
    title="Top 50 Providers: Claims vs Total Paid",
    labels={"total_claims": "Total Claims", "total_paid": "Total Paid ($)"},
)
fig.show()

## 5. Anomaly Detection — High Billing Flags
Flag providers whose `avg_paid_per_claim` is **3x above the national average** for their HCPCS code.

In [ ]:
national_avg = by_hcpcs.group_by("HCPCS_CODE").agg(
    pl.col("avg_paid_per_claim").mean().alias("national_avg")
)

# TODO: Join with top_providers on HCPCS_CODE and flag where
# provider avg_paid_per_claim > 3 * national_avg
# Export to '../exports/anomaly_flags.parquet'